In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [ ]:
def stumpf_S(z):
    """
    Stumpf function S(z).

    TODO: figure out what tolerance for |z|>tol I actually should use
    """
    if z > 1e-12:
        sqrt_z = np.sqrt(z)
        return (sqrt_z - np.sin(sqrt_z)) / (sqrt_z**3)
    elif z < -1e-12:
        sqrt_neg_z = np.sqrt(-z)
        return (np.sinh(sqrt_neg_z) - sqrt_neg_z) / (sqrt_neg_z**3)
    else:
        return 1/6

def stumpf_C(z):
    """
    Stumpf function C(z).

    TODO: figure out what tolerance for |z|>tol I actually should use
    """
    if z > 1e-12:
        sqrt_z = np.sqrt(z)
        return (1 - np.cos(sqrt_z)) / z
    elif z < -1e-12:
        sqrt_neg_z = np.sqrt(-z)
        return (np.cosh(sqrt_neg_z) - 1) / (-z)
    else:
        return 1/2

In [ ]:

def propagate_orbit(r0_vec, v0_vec, dt, mu):
    """
    Propagates an orbit using the universal variable formulation for Keplers problem.
    Fundamentals of Astrodynamics (BMWS), Sec 4.4.4.
    
    Args:
        r0_vec (array-like): Initial position vector (km)
        v0_vec (array-like): Initial velocity vector (km/s)
        dt (float): Time of flight (s)
        mu (float): Gravitational parameter (km^3/s^2)
        
    Returns:
        r_vec (np.array): Final position vector
        v_vec (np.array): Final velocity vector
    """
    # edge case
    if dt == 0:
        return np.array(r0_vec, dtype=float), np.array(v0_vec, dtype=float)

    # precompute some things that are used frequently
    r0_vec = np.array(r0_vec, dtype=float)
    v0_vec = np.array(v0_vec, dtype=float)
    
    r0 = np.linalg.norm(r0_vec)
    v0 = np.linalg.norm(v0_vec)
    rv0 = np.dot(r0_vec, v0_vec)

    # alpha = 1/a (a is semi-major axis)
    alpha = -v0**2/mu + 2/r0 
    
    # initial guess for chi (universal anomaly)
    if alpha > 1e-6:
        # ellipse (BMW eq. 4.75)
        chi = np.sqrt(mu) * dt * alpha
        print("DEBUG: ellipse: ", chi)

    elif alpha < -1e-6:
        # hyperbola (BMW eq. 4.76)
        # TODO: hyperbolas are broken right now, need to debug this further 
        chi = (
            np.sign(dt) * 
            np.sqrt(-1/alpha) * 
            np.log( 
                -2 * mu * dt * alpha / 
                (rv0 + np.sign(dt) * np.sqrt(-mu*alpha) * (1 - r0*alpha))
            )
        )
        print("DEBUG: hyperbola: ", chi)
        
    else:
        # parabola 
        #   1/6 chi^3 + sigma0/2 chi^2 + r0 chi = sqrt(mu) dt
        #   assume small step (linear term dominates)
        #   TODO: make a better guess
        chi = np.sqrt(mu) * dt / r0
        print("DEBUG: parabola: ", chi)

    # newton-raphson
    tol = 1e-8
    max_iter = 100
    counter = 0
    while(counter < max_iter):
        counter += 1

        # compute z from chi
        z = alpha * chi**2
        S = stumpf_S(z)
        C = stumpf_C(z)
        
        # universal kepler equation (BMW eq. 4.39 / 4.41)
        #   sqrt(mu)*dt = rv0/sqrt(mu) * chi^2 * C + (1 - r0*alpha) * chi^3 * S + r0 * chi
        #   f = sqrt(mu) * (tn - t)
        f = (
            (rv0 / np.sqrt(mu)) * chi**2 * C  
            + (1 - r0 * alpha) * chi**3 * S 
            + r0 * chi 
            - np.sqrt(mu) * dt
        )
        
        # universal variable formulation (BMW eq. 4.40 / 4.44)
        #   sqrt(mu) * dt/dx = r
        #                    = chi^2 * C + rv0/sqrt(mu) * chi * (1 - zS) + r0 * (1 - zC)
        fp = C * chi**2 + (rv0 / np.sqrt(mu)) * chi * (1 - z*S) + r0 * (1 - z*C) # (1 - alpha * r0) * chi**2 * C + r0
        

        # update chi (BMW eq. 4.42)
        dchi = f/fp
        chi = chi - dchi

        print(chi)
        
        if np.abs(dchi) < tol:
            break
    
    if counter == max_iter:
        print("Warning: max iterations reached when solving for x (universal anomaly)")

    # evaluate f and g functions, then compute r_vec
    f = 1 - (chi**2 / r0) * C            # (BMW eq. 4.58)
    g = dt - (chi**3 / np.sqrt(mu)) * S  # (BMW eq. 4.61)
    r_vec = f * r0_vec + g * v0_vec      # (BMW eq. 4.45)
    r = np.linalg.norm(r_vec)
    
    # evaluate fdot and gdot functions, then compute v_vec
    fdot = (np.sqrt(mu) / (r * r0)) * (alpha * chi**3 * S - chi)  # (BMW eq. 4.63)
    gdot = 1 - (chi**2 / r) * C                                   # (BMW eq. 4.62)
    v_vec = fdot * r0_vec + gdot * v0_vec                         # (BMW eq. 4.46)
    
    return r_vec, v_vec

In [ ]:
# consts
mu_earth = 398600.4418  # [km^3/s^2]
R_earth = 6378.0        # [km]

# geostationary
altitude = 35786.0                                   # [km]
velocity = np.sqrt(mu_earth / (R_earth + altitude))  # [km/s]
angle = np.pi / 6

# TESTING: convert to hyperbola
velocity = 1.5*velocity

r0 = np.array([R_earth + altitude, 0, 0])
v0 = np.array([0, velocity*np.cos(angle), velocity*np.sin(angle)])

# calculate semi-major axis a from vis-viva: v^2 = mu * (2/r - 1/a) => 1/a = 2/r - v^2/mu
r_mag = np.linalg.norm(r0)
v_mag = np.linalg.norm(v0)
energy = v_mag**2 / 2 - mu_earth / r_mag
a = -mu_earth / (2 * energy)

# propagate for one period
# period = 2 * np.pi * np.sqrt(a**3 / mu_earth)
period = 12*60*60
print(f"Period: {period:.2f} s")

r_final, v_final = propagate_orbit(r0, v0, period, mu_earth)

print("")
print("Initial r:", r0)
print("Final r:  ", r_final)
print("Difference r:", np.linalg.norm(r_final - r0))
print("")
print("Initial v:", v0)
print("Final v:  ", v_final)
print("Difference v:", np.linalg.norm(v_final - v0))

In [ ]:
times = np.linspace(0, period, 500)
positions = []

for t in times:
    r_t, _ = propagate_orbit(r0, v0, t, mu_earth)
    positions.append(r_t)
    
positions = np.array(positions)

In [ ]:
# plotting: matplotlib

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

# plot earth
u, v = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
x_earth = 6378 * np.cos(u) * np.sin(v)
y_earth = 6378 * np.sin(u) * np.sin(v)
z_earth = 6378 * np.cos(v)
ax.plot_surface(x_earth, y_earth, z_earth, color="blue", alpha=0.5)

# plot orbit
ax.plot(positions[:, 0], positions[:, 1], positions[:, 2], "k-", label="Orbit")
ax.scatter(positions[0, 0], positions[0, 1], positions[0, 2], c="g", marker="o", label="Start")
ax.scatter(positions[-1, 0], positions[-1, 1], positions[-1, 2], c="r", marker="x", label="End")

ax.set_xlabel("X (km)")
ax.set_ylabel("Y (km)")
ax.set_zlabel("Z (km)")
ax.set_title("Orbit")
ax.legend()

# set equal aspect ratio
max_range = np.array([positions[:, 0].max()-positions[:, 0].min(), 
                        positions[:, 1].max()-positions[:, 1].min(), 
                        positions[:, 2].max()-positions[:, 2].min()]).max() / 2.0

mid_x = (positions[:, 0].max()+positions[:, 0].min()) * 0.5
mid_y = (positions[:, 1].max()+positions[:, 1].min()) * 0.5
mid_z = (positions[:, 2].max()+positions[:, 2].min()) * 0.5

ax.set_xlim(mid_x - max_range, mid_x + max_range)
ax.set_ylim(mid_y - max_range, mid_y + max_range)
ax.set_zlim(mid_z - max_range, mid_z + max_range)


plt.tight_layout()
plt.show()

In [ ]:
# plotting: plotly

u, v = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
x_earth = 6378 * np.cos(u) * np.sin(v)
y_earth = 6378 * np.sin(u) * np.sin(v)
z_earth = 6378 * np.cos(v)

fig = go.Figure()

# add earth surface
fig.add_trace(go.Surface(
    x=x_earth, y=y_earth, z=z_earth,
    opacity=0.5,
    showscale=False, # hides the colorbar
    colorscale=[[0, "blue"], [1, "blue"]], # forces solid blue color
    name="Earth",
    hoverinfo="skip" # stops hover text on the planet surface
))

# add orbit trajectory
fig.add_trace(go.Scatter3d(
    x=positions[:, 0],
    y=positions[:, 1],
    z=positions[:, 2],
    mode="lines",
    line=dict(color="black", width=3),
    name="Orbit"
))

# add start point
fig.add_trace(go.Scatter3d(
    x=[positions[0, 0]],
    y=[positions[0, 1]],
    z=[positions[0, 2]],
    mode="markers",
    marker=dict(size=5, color="green", symbol="circle"),
    name="Start"
))

# add end point
fig.add_trace(go.Scatter3d(
    x=[positions[-1, 0]],
    y=[positions[-1, 1]],
    z=[positions[-1, 2]],
    mode="markers",
    marker=dict(size=5, color="red", symbol="x"),
    name="End"
))

# layout Settings
fig.update_layout(
    title="Orbit",
    scene=dict(
        xaxis_title="X (km)",
        yaxis_title="Y (km)",
        zaxis_title="Z (km)",
        # "data" aspectmode automatically fixes the aspect ratio so earth doesn't look like an egg
        aspectmode="data" 
    ),
    margin=dict(l=0, r=0, b=0, t=40) # tighter layout
)

fig.show()